# UR5 equations of motion — three forward-dynamics pipelines

Textbook-style comparison of how minilink evaluates UR5 joint accelerations $\ddot q$ from

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau.
$$

Sections **1–4** derive the mathematics of each pipeline (RNEA–$H$, ABA, symbolic Lagrange). Sections **5–7** validate single-step forward dynamics numerically; section **8** integrates a full rollout.

| Pipeline | Idea | minilink entry |
| --- | --- | --- |
| **RNEA–$H$** | RNEA bias + explicit inertia solve | `UR5Manipulator.forward_dynamics_rnea_h` |
| **ABA** | Articulated-body algorithm ($O(n)$ spatial) | `UR5Manipulator.forward_dynamics` (catalog default) |
| **Symbolic Lagrange** | Derive $H,C,g$ once; lambdify | `minilink.symbolic` → `to_minilink()` |

Helpers: `compare_eom.py`, `rollout_compare.py`, `symbolic_ur5.py`. Thin CLI smoke: `run_demo.py`.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

repo = Path.cwd()
if not (repo / "minilink").is_dir():
    repo = repo.parents[2]
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from examples.projects.ur5_dynamics.compare_eom import (
    DEFAULT_N_SAMPLES,
    DEFAULT_N_TIMING,
    DEFAULT_SEED,
    aba_speedup,
    accuracy_summary,
    build_evaluation_batch,
    comprehensive_comparison,
    forward_dynamics_aba,
    forward_dynamics_rnea_h,
    forward_dynamics_symbolic,
    named_case_summary,
    plot_comparison,
    plot_per_joint_errors,
)
from examples.projects.ur5_dynamics.rollout_compare import (
    DEFAULT_ROLLOUT_BACKEND,
    DEFAULT_ROLLOUT_DT,
    DEFAULT_ROLLOUT_TF,
    plot_rollout_timings,
    rollout_comparison,
    rollout_error_rows,
    rollout_timing_rows,
)
from examples.projects.ur5_dynamics.symbolic_ur5 import (
    build_symbolic_ur5,
    catalog_params_no_damping,
)
from minilink.dynamics.catalog.manipulators.ur5 import UR5Manipulator

%matplotlib inline

## 1. Manipulator equation of motion

Consider a serial $n$-joint manipulator in generalized coordinates $q \in \mathbb{R}^n$. The standard second-order model is

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau,
$$

where $H(q) \in \mathbb{R}^{n \times n}$ is the symmetric positive-definite inertia matrix, $C(q,\dot q)\dot q$ collects Coriolis and centrifugal terms, $g(q)$ is gravity, $d$ is dissipation, and $\tau$ is applied joint torque.

Define the **bias force** (everything that is not inertia times acceleration):

$$
b(q,\dot q) = C(q,\dot q)\,\dot q + g(q).
$$

**Forward dynamics** solves for the generalized acceleration $\ddot q$ given $(q,\dot q,\tau)$:

$$
\ddot q = H(q)^{-1}\bigl(\tau - b(q,\dot q) - d(q,\dot q)\bigr).
$$

For simulation, minilink stacks state $x = [q;\dot q] \in \mathbb{R}^{2n}$ so that

$$
\dot x = \begin{bmatrix} \dot q \\ \ddot q \end{bmatrix} = f(x,u) = \begin{bmatrix} \dot q \\ \text{FD}(q,\dot q,u) \end{bmatrix}.
$$

On the UR5 catalog plant, **`H`**, **`C`**, and **`g`** are built from spatial RNEA; the default **`forward_dynamics`** uses the Articulated-Body Algorithm (ABA). Below we **disable viscous damping** ($d=0$) so the symbolic Lagrange export matches the spatial parameters.


## 2. Pipeline A — Recursive Newton–Euler + explicit $H$ (RNEA–$H$)

### 2.1 Spatial vectors

Each link $i$ carries a 6-vector spatial velocity $v_i = \begin{bmatrix} \omega_i \\ v_i \end{bmatrix}$ and spatial force $f_i$. The **spatial inertia** $I_i$ maps acceleration to force. The **motion cross** operator $\mathrm{crm}(v)$ and its dual $\mathrm{crm}(v)^\top$ appear in the force balance

$$
f_i = I_i a_i - \mathrm{crm}(v_i)^\top I_i v_i.
$$

### 2.2 Inverse dynamics (one tree pass each way)

Given $(q,\dot q,\ddot q)$, RNEA computes $\tau$ in two sweeps:

**Outward** (base $\to$ tip): for each joint $i$,

$$
v_i = {}^{i}\!X_{i-1}\, v_{i-1} + S_i\,\dot q_i, \qquad
a_i = {}^{i}\!X_{i-1}\, a_{i-1} + S_i\,\ddot q_i + \mathrm{crm}(v_i)\,S_i\,\dot q_i,
$$

then accumulate $f_i$ from $I_i$, $a_i$, and $v_i$.

**Inward** (tip $\to$ base): project forces onto joints,

$$
\tau_i = S_i^\top f_i, \qquad f_{i-1} \mathrel{+}= {}^{i}\!X_{i-1}^\top f_i.
$$

We write $\tau = \mathrm{RNEA}(q,\dot q,\ddot q)$.

### 2.3 Building $H$ and forward dynamics

The bias force at zero acceleration is

$$
b(q,\dot q) = \mathrm{RNEA}(q,\dot q, 0).
$$

Each column of the inertia matrix is one inverse-dynamics call with a unit joint acceleration:

$$
H_{:,j}(q) = \mathrm{RNEA}(q, 0, e_j), \qquad j = 1,\ldots,n,
$$

followed by symmetrization $H \leftarrow \tfrac12(H + H^\top)$. Forward dynamics is the linear solve

$$
\ddot q = H^{-1}(\tau - b - d).
$$

**Complexity:** one bias pass $O(n)$, $n$ columns $O(n^2)$, solve $O(n^3)$ — for UR5 ($n=6$) the solve is negligible; forming $H$ dominates.

**minilink:** `UR5Manipulator.forward_dynamics_rnea_h`, and `H` / `g` / `C` via the same spatial RNEA stack.


## 3. Pipeline B — Articulated Body Algorithm (ABA)

ABA computes the **same** $\ddot q$ as RNEA–$H$ but never assembles $H(q)$. It maintains articulated-body inertias $I_i^A$ and bias forces $p_{A,i}$ while propagating along the kinematic tree.

### 3.1 Pass 1 — outward (velocities and bias)

For each link $i$, with joint motion subspace $S_i$ and parent transform ${}^{i}\!X_{i-1}$:

$$
v_i = {}^{i}\!X_{i-1}\, v_{i-1} + S_i\,\dot q_i, \qquad
c_i = \mathrm{crm}(v_i)\,S_i\,\dot q_i,
$$

$$
p_{A,i} = -\mathrm{crm}(v_i)^\top I_i v_i.
$$

### 3.2 Pass 2 — inward (articulated inertia)

Initialize $I_i^A = I_i$. From tip to base, for each $i$:

$$
U_i = I_i^A S_i, \qquad d_i = S_i^\top U_i, \qquad u_i = \tau_i - S_i^\top p_{A,i},
$$

$$
I_i^A \leftarrow I_i^A - \frac{U_i U_i^\top}{d_i}, \qquad
p_{A,i-1} \mathrel{+}= {}^{i}\!X_{i-1}^\top\!\left(p_{A,i} + I_i^A c_i + \frac{U_i u_i}{d_i}\right),
$$

with the articulated inertia $I_i^A$ propagated to the parent before processing the next link.

### 3.3 Pass 3 — outward (accelerations)

Starting from the base spatial acceleration $a_0$ (gravity), for each $i$:

$$
a_i = {}^{i}\!X_{i-1}\, a_{i-1} + c_i, \qquad
\ddot q_i = \frac{u_i - U_i^\top a_i}{d_i}, \qquad
a_i \leftarrow a_i + S_i\,\ddot q_i.
$$

**Complexity:** each pass is $O(n)$, so **$O(n)$ per forward-dynamics call**.

**minilink:** `UR5Manipulator.forward_dynamics` (catalog default for simulation and `f`).


## 4. Pipeline C — Symbolic Lagrange derivation

### 4.1 Energies

Build a DH chain in SymPy. With kinetic energy $T(q,\dot q)$ and potential $V(q)$,

$$
L(q,\dot q) = T(q,\dot q) - V(q).
$$

For rigid links, $T = \sum_i \tfrac12 v_{c,i}^\top M_i v_{c,i} + \tfrac12 \omega_i^\top I_i \omega_i$ expressed in joint coordinates.

### 4.2 Euler–Lagrange equations

$$
\frac{d}{dt}\frac{\partial L}{\partial \dot q} - \frac{\partial L}{\partial q} = \tau.
$$

Expanding the $\dot q$-dependent terms yields the standard manipulator form

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + g(q) = \tau,
$$

where

$$
H_{ij} = \frac{\partial^2 T}{\partial \dot q_i \partial \dot q_j}, \qquad
g_i = \frac{\partial V}{\partial q_i},
$$

and the Coriolis matrix $C$ follows from Christoffel symbols of $H(q)$ (or equivalent Kane/Lagrange bookkeeping).

### 4.3 Export and evaluation

SymPy derives $H(q)$, $C(q,\dot q)$, and $g(q)$ symbolically. `to_minilink()` **lambdifies** them once into numeric callables; each forward-dynamics step evaluates $H$ and solves

$$
\ddot q = H^{-1}(\tau - C\dot q - g - d),
$$

so runtime cost is similar to RNEA–$H$, while the **upfront** derive + export cost is large (minutes for UR5).

**minilink:** `minilink.symbolic.mechanics` → `build_symbolic_ur5()` in this project.


### 4.4 Complexity summary

| Pipeline | Dominant per-step cost | Forms $H(q)$? |
| --- | --- | --- |
| RNEA–$H$ | $O(n^2)$ from $n$ RNEA columns | yes |
| ABA | $O(n)$ three tree passes | no |
| Symbolic Lagrange (numeric) | $O(n^2)$ evaluate $H$ + $O(n^3)$ solve | yes |

All three target the same $\ddot q$ when the underlying model data match; they differ in **algorithm** and **when** work is paid (symbolic: mostly upfront).

## 5. Hands-on — one configuration

Before the batch study, call each pipeline on the same $(q, \dot q, \tau)$.


In [ ]:
arm = UR5Manipulator()
params = catalog_params_no_damping()

q = np.array([0.1, -0.5, 0.2, -1.0, 0.3, 0.0])
v = np.array([0.5, -0.3, 0.2, 0.1, -0.4, 0.2])
u = np.zeros(6)

qdd_rnea = forward_dynamics_rnea_h(arm, q, v, u, params)
qdd_aba = forward_dynamics_aba(arm, q, v, u, params)

print("RNEA–H qdd:", np.array2string(qdd_rnea, precision=4, suppress_small=True))
print("ABA    qdd:", np.array2string(qdd_aba, precision=4, suppress_small=True))
print("max |Δqdd| (ABA vs RNEA–H):", np.max(np.abs(qdd_rnea - qdd_aba)))

In [ ]:
# Symbolic plant: first call ~2–3 min; cached afterward.
# Set os.environ["UR5_SKIP_SYMBOLIC"] = "1" before this cell for an ABA-only run.
symbolic_plant = None
if os.environ.get("UR5_SKIP_SYMBOLIC", "").strip().lower() not in {"1", "true", "yes"}:
    try:
        symbolic_plant = build_symbolic_ur5(verbose=True)
        qdd_sym = forward_dynamics_symbolic(symbolic_plant, q, v, u)
        print("Symbolic qdd:", np.array2string(qdd_sym, precision=4, suppress_small=True))
        print("max |Δqdd| (Symbolic vs RNEA–H):", np.max(np.abs(qdd_rnea - qdd_sym)))
    except ImportError:
        print("SymPy not installed — pip install minilink[symbolic]")
else:
    print("UR5_SKIP_SYMBOLIC set — skipping symbolic pipeline.")

## 6. Single-step batch study — accuracy and timing

**67 samples:** three named poses plus **64 random** draws with **seed 0** (reproducible). Reference: RNEA–$H$.


In [ ]:
SEED = DEFAULT_SEED
N_RANDOM = DEFAULT_N_SAMPLES
N_TIMING = DEFAULT_N_TIMING

configs, labels = build_evaluation_batch(seed=SEED, n_random=N_RANDOM)
result = comprehensive_comparison(
    arm,
    symbolic_plant,
    configs,
    params=params,
    seed=SEED,
    n_timing=N_TIMING,
    case_labels=labels,
)

print(f"Batch: {result.n_samples} samples ({N_RANDOM} random + 3 named, seed={SEED})")

In [ ]:
def _rows_to_markdown(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        cells = []
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                if col.endswith("_ms"):
                    cells.append(f"{val:.3f}")
                elif val == 0.0:
                    cells.append("0")
                else:
                    cells.append(f"{val:.3e}")
            else:
                cells.append(str(val))
        body.append("| " + " | ".join(cells) + " |")
    return "\n".join([header, sep] + body)

acc_cols = [
    "method",
    "max_abs_dqdd",
    "mean_abs_dqdd",
    "rms_abs_dqdd",
    "p95_abs_dqdd",
    "median_ms",
]
display(Markdown("### Accuracy and timing (vs RNEA–H reference)"))
display(Markdown(_rows_to_markdown(accuracy_summary(result), acc_cols)))

case_cols = list(named_case_summary(result)[0].keys())
display(Markdown("### Named case studies"))
display(Markdown(_rows_to_markdown(named_case_summary(result), case_cols)))

display(Markdown(f"**ABA median speedup vs RNEA–H:** {aba_speedup(result):.2f}×"))

In [ ]:
fig, _ = plot_comparison(result)
fig.suptitle("UR5 forward dynamics — random-batch accuracy and timing", y=1.02)
plt.show()

In [ ]:
if "Symbolic" in result.methods:
    fig2, _ = plot_per_joint_errors(result)
    plt.show()
else:
    print("Per-joint symbolic plot skipped (symbolic plant not built).")

## 8. Rollout simulation — integrating $\dot x = f(x,u)$

Forward dynamics returns one $\ddot q$; a **rollout** applies it on every step of a time grid $t_k = k\,\Delta t$.

Stack $x = [q;\dot q]$. With fixed-step RK4 and constant input $\bar u$,

$$
x_{k+1} = \Phi_{\Delta t}(x_k, \bar u), \qquad k = 0,1,\ldots,N-1,
$$

where each step uses four evaluations of

$$
f(x,u) = \begin{bmatrix} \dot q \\ \text{FD}(q,\dot q,u) \end{bmatrix}.
$$

The catalog UR5 uses **ABA** inside $\text{FD}$; we compare integrated trajectories to a **RNEA–$H$** plant on the same grid ($t_f = 1\,\mathrm{s}$, $\Delta t = 2\,\mathrm{ms}$, zero torque, `compile_backend="jax"`).

**Timing phases** (wall clock, per pipeline):

| Phase | What it measures |
| --- | --- |
| **compile** | `Simulator` construction (graph build) |
| **JIT warm-up** | first `solve()` — JAX RK4 scan compile + first run |
| **rollout** | second `solve()` — integration only |

Reference: **RNEA–$H$** trajectory $\{x_k^{\mathrm{ref}}\}$. Report $\max_k \|x_k - x_k^{\mathrm{ref}}\|$ for ABA and symbolic rollouts.

In [ ]:
# JAX-exported symbolic plant for rollout (cached separately from NumPy FD plant).
symbolic_rollout = None
if symbolic_plant is not None:
    symbolic_rollout = build_symbolic_ur5(backend=DEFAULT_ROLLOUT_BACKEND, verbose=False)

rollout_result, ref_traj = rollout_comparison(
    params,
    symbolic_plant=symbolic_rollout,
    compile_backend=DEFAULT_ROLLOUT_BACKEND,
    tf=DEFAULT_ROLLOUT_TF,
    dt=DEFAULT_ROLLOUT_DT,
)
print(
    f"Rollout grid: tf={rollout_result.tf}s, dt={rollout_result.dt}s, "
    f"n={rollout_result.n_steps}, backend={rollout_result.backend}"
)

In [ ]:
timing_cols = ["method", "compile_ms", "jit_warmup_ms", "rollout_ms", "backend"]
display(Markdown("### Rollout timings"))
display(Markdown(_rows_to_markdown(rollout_timing_rows(rollout_result), timing_cols)))

if rollout_result.max_state_error:
    err_cols = list(rollout_error_rows(rollout_result)[0].keys())
    display(Markdown("### Trajectory error vs RNEA–H"))
    display(Markdown(_rows_to_markdown(rollout_error_rows(rollout_result), err_cols)))

In [ ]:
fig3, _ = plot_rollout_timings(rollout_result)
plt.show()

## 9. Summary

* **Single-step FD:** ABA and symbolic Lagrange match RNEA–$H$ to near machine precision on the full batch ($\max |\Delta \ddot q| \lesssim 10^{-10}$ rad/s$^2$ typically).
* **Rollout:** integrated trajectories agree to similar tolerance when JAX uses float64; **JIT warm-up** is paid once, then **rollout** timing reflects pure integration.
* **Speed:** ABA wins both per FD call and per rollout step count because it never forms $H$.

**When to use which:** ABA for simulation loops on long chains; RNEA–$H$ (or exported symbolic $H$, $g$) for teaching, linearization, and control design.
